# Transformer Tabanlı Yapay Zekâ Metni Tespiti
### Dayanıklılık ve Açıklanabilirlik Analizi

**Ders projesi:** Perseptron Ağlar ve Uygulamaları (BM466)  
**Hazırlayan:** Fundanur Öztürk  
**Yarışma:** Kaggle — LLM: Detect AI Generated Text

---

Bu not defteri üç katmandan oluşur:
1. **Tespit** — RoBERTa-base fine-tuning ile ikili sınıflandırma
2. **Dayanıklılık (Robustness)** — adversarial saldırılar altında test
3. **Açıklanabilirlik (Explainability)** — SHAP ile karar analizi

Son adımda canlı demo için **Gradio** arayüzü oluşturulur.

## 0. Kurulum ve Kütüphaneler

In [ ]:
# Gerekli ek kütüphaneleri kur
!pip install -q nlpaug shap gradio sacremoses

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, classification_report
import warnings
warnings.filterwarnings("ignore")

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Cihaz:", device)
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "yok")

## 1. Veri Setinin Yüklenmesi

İki kaynağı birleştiriyoruz:
- **Kaggle LLM-Detect** — yarışmanın resmi verisi (1.378 metin, sadece 3 AI)
- **DAIGT V2** — açık kaynak genişletilmiş derlem (~44.868 metin)

Toplam: **~46.246** metin

In [ ]:
# Doğru Kaggle dataset yolları
COMP_PATH  = "/kaggle/input/competitions/llm-detect-ai-generated-text/train_essays.csv"
DAIGT_PATH = "/kaggle/input/datasets/thedrcat/daigt-v2-train-dataset/train_v2_drcat_02.csv"

comp = pd.read_csv(COMP_PATH)
comp = comp[["text", "generated"]].rename(columns={"generated": "label"})
print("Yarışma verisi:", comp.shape)
print(comp["label"].value_counts())

In [ ]:
daigt = pd.read_csv(DAIGT_PATH)
daigt = daigt[["text", "label"]]
print("DAIGT V2:", daigt.shape)
print(daigt["label"].value_counts())

In [ ]:
df = pd.concat([comp, daigt], ignore_index=True)
df = df.dropna(subset=["text"]).drop_duplicates(subset=["text"]).reset_index(drop=True)
df["label"] = df["label"].astype(int)
print("Birlestirilmis:", df.shape)
print(df["label"].value_counts())
print("\n0 = Insan, 1 = Yapay Zeka")

## 2. Keşifsel Veri Analizi (EDA)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

counts = df["label"].value_counts().sort_index()
axes[0].bar(["Insan (0)", "Yapay Zeka (1)"], counts.values, color=["#2E6DA4", "#E07A5F"])
axes[0].set_title("Sinif Dagilimi")
axes[0].set_ylabel("Metin Sayisi")
for i, v in enumerate(counts.values):
    axes[0].text(i, v, f"{v:,}", ha="center", va="bottom")

df["n_words"] = df["text"].str.split().str.len()
axes[1].hist(df[df.label==0]["n_words"], bins=50, alpha=0.6, label="Insan",
             color="#2E6DA4", range=(0,800))
axes[1].hist(df[df.label==1]["n_words"], bins=50, alpha=0.6, label="Yapay Zeka",
             color="#E07A5F", range=(0,800))
axes[1].set_title("Metin Uzunlugu Dagilimi (kelime)")
axes[1].set_xlabel("Kelime sayisi")
axes[1].legend()

plt.tight_layout()
plt.show()

## 3. Eğitim / Test Ayrımı

Hız için dengeli bir alt küme: **10.000 eğitim / 2.500 test**  
Stratified split, seed=42 (tekrarlanabilirlik)

In [ ]:
n_per_class = 6250
sub = (df.groupby("label", group_keys=False)
         .apply(lambda x: x.sample(min(len(x), n_per_class), random_state=SEED))
         .reset_index(drop=True))

train_df, test_df = train_test_split(
    sub, test_size=2500, stratify=sub["label"], random_state=SEED
)
train_df = train_df.reset_index(drop=True)
test_df  = test_df.reset_index(drop=True)

print("Egitim:", train_df.shape, "| Test:", test_df.shape)
print("Egitim sinif:", train_df["label"].value_counts().to_dict())
print("Test sinif:  ", test_df["label"].value_counts().to_dict())

## 4. Tokenizasyon ve Dataset Sınıfı

RoBERTa'nın BPE tabanlı tokenizer'ı — azami 128 token.

In [ ]:
from transformers import AutoTokenizer

MODEL_NAME = "roberta-base"
MAX_LEN    = 128

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

class TextDataset(torch.utils.data.Dataset):
    def __init__(self, texts, labels, tokenizer, max_len=MAX_LEN):
        self.texts     = list(texts)
        self.labels    = list(labels)
        self.tokenizer = tokenizer
        self.max_len   = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        enc = self.tokenizer(
            self.texts[idx], truncation=True,
            padding="max_length", max_length=self.max_len,
            return_tensors="pt",
        )
        item = {k: v.squeeze(0) for k, v in enc.items()}
        item["labels"] = torch.tensor(self.labels[idx], dtype=torch.long)
        return item

train_ds = TextDataset(train_df["text"], train_df["label"], tokenizer)
test_ds  = TextDataset(test_df["text"],  test_df["label"],  tokenizer)
print("Dataset hazir:", list(train_ds[0].keys()))

## 5. Modelin İnce Ayarı (Fine-tuning)

RoBERTa-base (~124M parametre) transfer öğrenme ile eğitiliyor.  
Hiperparametreler: **2 epoch, batch=32, lr=2e-5, fp16**  
⏱️ Yaklaşık 5 dakika sürer.

In [ ]:
from transformers import AutoModelForSequenceClassification, TrainingArguments, Trainer

model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        "accuracy": accuracy_score(labels, preds),
        "f1": f1_score(labels, preds),
    }

args = TrainingArguments(
    output_dir="/kaggle/working/results",
    num_train_epochs=2,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=64,
    learning_rate=2e-5,
    weight_decay=0.01,
    fp16=torch.cuda.is_available(),
    eval_strategy="epoch",
    save_strategy="no",
    logging_steps=50,
    report_to="none",
    seed=SEED,
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=train_ds,
    eval_dataset=test_ds,
    compute_metrics=compute_metrics,
)

trainer.train()

## 6. Değerlendirme

Test kümesinde doğruluk, F1 ve confusion matrix.  
**Ana bulgu:** False positive >> False negative (insan metni AI sanılıyor)

In [ ]:
pred_output = trainer.predict(test_ds)
y_pred = np.argmax(pred_output.predictions, axis=-1)
y_true = test_df["label"].values

print(f"Dogruluk : {accuracy_score(y_true, y_pred):.4f}")
print(f"F1-skoru : {f1_score(y_true, y_pred):.4f}\n")
print(classification_report(y_true, y_pred, target_names=["Insan", "Yapay Zeka"]))

In [ ]:
cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(7, 5.5))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=["Insan", "AI"], yticklabels=["Insan", "AI"],
            annot_kws={"size": 16, "weight": "bold"})
plt.title("Confusion Matrix - RoBERTa AI Metin Tespiti\n(Test seti: 2500 ornek)")
plt.ylabel("Gercek Etiket")
plt.xlabel("Tahmin Edilen Etiket")
plt.tight_layout()
plt.savefig("/kaggle/working/confusion_matrix.png", dpi=120)
plt.show()

tn, fp, fn, tp = cm.ravel()
print(f"Dogru Negatif  (Insan->Insan): {tn}")
print(f"Yanlis Pozitif (Insan->AI)   : {fp}  <- ana bulgu!")
print(f"Yanlis Negatif (AI->Insan)   : {fn}")
print(f"Dogru Pozitif  (AI->AI)      : {tp}")

## 7. Dayanıklılık (Robustness) Analizi

100 AI metni üzerinde 3 senaryo:
1. **Orijinal** (kontrol)
2. **Synonym** — kelimelerin %30'u eş anlamlısıyla değiştirilir
3. **Typo** — karakter düzeyi yazım hatası eklenir

In [ ]:
import nltk
nltk.download('averaged_perceptron_tagger_eng')

import nlpaug.augmenter.word as naw
import nlpaug.augmenter.char as nac

syn_aug  = naw.SynonymAug(aug_src="wordnet", aug_p=0.3)
typo_aug = nac.RandomCharAug(action="swap", aug_char_p=0.1, aug_word_p=0.3)
ai_texts = test_df[test_df["label"]==1]["text"].sample(100, random_state=SEED).tolist()

@torch.no_grad()
def predict_batch(texts):
    model.eval()
    enc = tokenizer(texts, truncation=True, padding=True,
                    max_length=MAX_LEN, return_tensors="pt").to(device)
    probs = torch.softmax(model(**enc).logits, dim=-1)[:, 1]
    preds = (probs >= 0.5).long()
    return preds.cpu().numpy(), probs.cpu().numpy()

scenarios = {
    "Orijinal": ai_texts,
    "Synonym":  [syn_aug.augment(t)[0]  for t in ai_texts],
    "Typo":     [typo_aug.augment(t)[0] for t in ai_texts],
}

results = {}
for name, texts in scenarios.items():
    preds, probs = predict_batch(texts)
    correct = int((preds == 1).sum())
    results[name] = {"correct": correct, "avg_prob": float(np.mean(probs))}
    print(f"{name:10s}: {correct}/100 | ort. AI prob = {np.mean(probs):.4f}")

In [ ]:
names    = list(results.keys())
corrects = [results[n]["correct"] for n in names]
colors   = ["#1f77b4", "#ff7f0e", "#d62728"]

plt.figure(figsize=(9, 5.5))
bars = plt.bar(names, corrects, color=colors, edgecolor="black")
plt.axhline(100, ls="--", color="green", alpha=0.6, label="Maksimum (100)")
plt.ylim(0, 110)
plt.ylabel("Dogru Tahmin Sayisi (100 uzerinden)")
plt.title("Robustness Analizi: Adversarial Saldirilar Karsisinda Performans\n(100 AI metni uzerinde)")
for b, n in zip(bars, names):
    plt.text(b.get_x()+b.get_width()/2, b.get_height()+1,
             f"{results[n]['correct']}/100\n(ort. prob: {results[n]['avg_prob']:.4f})",
             ha="center", va="bottom", fontweight="bold")
plt.legend(loc="lower right")
plt.tight_layout()
plt.savefig("/kaggle/working/robustness_analysis.png", dpi=120)
plt.show()

## 8. Açıklanabilirlik (SHAP) Analizi

SHAP ile her kelimenin karara katkısını ölçüyoruz.  
**Beklenen bulgu:** "important", "significant", "particular" gibi **akademik kalıp kelimeler** AI sinyali veriyor → distributional bias.

In [ ]:
import shap

@torch.no_grad()
def f_predict(texts):
    model.eval()
    texts = [str(t) for t in texts]
    enc = tokenizer(list(texts), truncation=True, padding=True,
                    max_length=MAX_LEN, return_tensors="pt").to(device)
    return torch.softmax(model(**enc).logits, dim=-1)[:, 1].cpu().numpy()

# Resmi/akademik metin - sunumdaki kelimeleri verir
sample = (
    "In particular, the impact of human activities on the environment has become "
    "a significant concern. It is important to recognize that our actions have a "
    "profound effect on the world around us. Climate change, deforestation, and "
    "pollution are critical issues that demand immediate attention and significant "
    "action from all members of society. The effect of these actions on future "
    "generations is an important consideration that cannot be overlooked."
)

masker      = shap.maskers.Text(tokenizer)
explainer   = shap.Explainer(f_predict, masker)
shap_values = explainer([sample])
print("SHAP hesaplandi!")
print(f"Model tahmini: %{f_predict([sample])[0]*100:.2f} AI")

In [ ]:
words = shap_values.data[0]
vals  = shap_values.values[0]
pairs = [(w.strip(), float(v)) for w, v in zip(words, vals) if w.strip()]
top   = sorted(pairs, key=lambda x: abs(x[1]), reverse=True)[:15]

print(f"{'Sira':<5}{'Kelime':<18}{'SHAP':>10}  Yon")
print("-"*42)
for i, (w, v) in enumerate(top, 1):
    yon = "AI sinyali" if v > 0 else "Insan sinyali"
    print(f"{i:<5}{w:<18}{v:>+.4f}  {yon}")

plt.figure(figsize=(9, 7))
tw = [w for w, _ in top][::-1]
tv = [v for _, v in top][::-1]
plt.barh(range(len(tw)), tv,
         color=["#e8413a" if v > 0 else "#1f77b4" for v in tv])
plt.yticks(range(len(tw)), tw)
plt.xlabel("SHAP Degeri (pozitif = AI sinyali, negatif = Insan sinyali)")
plt.title("SHAP Analizi: En Etkili 15 Kelime\n(Test metni: AI uretimi, model %99.96 AI dedi)")
plt.tight_layout()
plt.savefig("/kaggle/working/shap_analysis.png", dpi=120)
plt.show()

## 9. Modelin Kaydedilmesi

In [ ]:
SAVE_DIR = "/kaggle/working/roberta_model"
model.save_pretrained(SAVE_DIR)
tokenizer.save_pretrained(SAVE_DIR)
print("Model kaydedildi ->", SAVE_DIR)
print("Dosyalar:", os.listdir(SAVE_DIR))

## 10. Çıkarım (Inference) Fonksiyonu

### Distributional Bias Kanıtı
3 farklı insan metni → hepsi AI Uretimi çıkıyor!

In [ ]:
@torch.no_grad()
def full_predict(text):
    if not text or len(text.strip()) < 10:
        return "Kisa metin!", {}
    model.eval()
    enc = tokenizer(text, truncation=True, padding=True,
                    max_length=MAX_LEN, return_tensors="pt").to(device)
    probs  = torch.softmax(model(**enc).logits, dim=-1)[0].cpu().numpy()
    p_h, p_ai = float(probs[0]), float(probs[1])
    label  = "AI Uretimi" if p_ai >= 0.5 else "Insan Yazimi"
    return f"{label} (Guven: %{max(p_h,p_ai)*100:.2f})", {"AI Uretimi": p_ai, "Insan Yazimi": p_h}

print("=== DISTRIBUTIONAL BIAS KANITLARI ===\n")
testler = [
    ("RESMI INSAN YAZISI",
     "In particular, the impact of human activities on the environment has become a significant concern."),
    ("GUNLUK INSAN YAZISI",
     "dude i swear my cat just knocked over my coffee AGAIN like the third time this week lol"),
    ("BILIMSEL INSAN YAZISI",
     "The mitochondria is the powerhouse of the cell. It generates ATP for the cell."),
]
for tip, metin in testler:
    sonuc, _ = full_predict(metin)
    print(f"[{tip}]")
    print(f"Metin : {metin[:75]}")
    print(f"Sonuc : {sonuc}\n")

## 11. Gradio Arayüzü (Canlı Demo)

`share=True` ile herkese açık geçici URL oluşturur (72 saat aktif).

In [ ]:
import gradio as gr

EXAMPLES = [
    "In particular, the impact of human activities on the environment has become a significant concern.",
    "dude i swear my cat just knocked over my coffee AGAIN like the third time this week lol",
    "The mitochondria is the powerhouse of the cell.",
]

with gr.Blocks(theme=gr.themes.Soft()) as demo:
    gr.Markdown("# 🔍 AI Metin Tespiti (RoBERTa Fine-tuned)")
    gr.Markdown(
        "**Model:** RoBERTa-base (124M parametre) &nbsp;·&nbsp; "
        "**Dogruluk:** ~%95 &nbsp;·&nbsp; "
        "**Proje:** Perseptron Aglar ve Uygulamalari dersi"
    )
    with gr.Row():
        inp = gr.Textbox(label="Analiz Edilecek Metin", lines=6,
                         placeholder="Metni buraya yapistirin...")
        with gr.Column():
            out_label = gr.Textbox(label="Sonuc")
            out_plot  = gr.Label(label="Olasilik Dagilimi")
    btn = gr.Button("🔎 Analiz Et", variant="primary")
    gr.Examples(EXAMPLES, inputs=inp, label="Hazir Ornekler")
    gr.Markdown(
        "**Not:** Sonuclar olasiliksaldir. "
        "Model resmi/akademik kalip kelimelere AI sinyali olarak tepki vermektedir "
        "(SHAP analizi — distributional bias bulgusudur)."
    )
    btn.click(full_predict, inputs=inp, outputs=[out_label, out_plot])

demo.launch(share=True)

## 12. Özet

| Aşama | Sonuç |
|-------|-------|
| **Sınıflandırma** | ~%94,7 doğruluk, F1 ≈ 0.93 |
| **Hata yapısı** | FP >> FN — insan metni AI sanılıyor |
| **Dayanıklılık** | Synonym + Typo saldırılarında 99/100 |
| **SHAP bulgusu** | "important", "significant", "particular" → AI sinyali |

### Ana Bulgu — Distributional Bias
Model **"yapay zekâ mı, insan mı"** değil, **"resmî yazılmış mı"** sorusuna cevap veriyor.
Yüksek doğruluk tek başına yeterli değildir.